# Model Parallelism: A Tiny 2-Layer ANN Split Across CPU + GPU

**My setup:** Intel i5 (CPU) + NVIDIA GTX 1650 (1 CUDA GPU).

Since there is only **one CUDA GPU** here, not two, we can't do the classic "GPU 0 + GPU 1" split.
Instead, this notebook does the *same idea* (Model Parallelism, see file `03-model-parallelism.md`)
using the two compute devices you actually have:

- **Device A = `cpu`** (your Intel i5) → holds **Layer 1**
- **Device B = `cuda:0`** (your GTX 1650) → holds **Layer 2**

The activations produced by Layer 1 (on CPU) are sent over to Layer 2 (on GPU) during the forward
pass, and gradients flow back the other way during the backward pass — exactly like the
"activations flow from GPU 1 to GPU 2" diagram in the docs, just with CPU standing in for GPU 1.

> If you later get a second CUDA GPU, you only need to change two lines (`device_a`, `device_b`)
> to `cuda:0` and `cuda:1` — nothing else in this notebook changes.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


PyTorch version: 2.5.1+cu121
CUDA available: True
GPU name: NVIDIA GeForce GTX 1650


## Step 1: Pick the Two Devices

**Keywords:** `device` = the piece of hardware a tensor lives on and computes on (CPU or a specific GPU).

- `device_a` — where Layer 1 will live (CPU, your i5)
- `device_b` — where Layer 2 will live (GPU, your GTX 1650)


In [2]:
device_a = torch.device("cpu")
device_b = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

print("Layer 1 will run on:", device_a)
print("Layer 2 will run on:", device_b)


Layer 1 will run on: cpu
Layer 2 will run on: cuda:0


## Step 2: Define the Model, Split Across Devices

This is a very small 2-layer ANN:

- **Layer 1**: Linear(in=4, out=8) + ReLU — placed on `device_a` (CPU)
- **Layer 2**: Linear(in=8, out=1) — placed on `device_b` (GPU)

Notice in `forward()`: after Layer 1 finishes on the CPU, we manually move (`.to(device_b)`)
the output to the GPU before feeding it into Layer 2. This one line **is** model parallelism —
it's the "activations flow from one device to the next" step from the docs.


In [3]:
class TwoLayerModelParallelANN(nn.Module):
    def __init__(self, device_a, device_b):
        super().__init__()
        self.device_a = device_a
        self.device_b = device_b

        # Stage 1: lives on device_a (CPU)
        self.layer1 = nn.Sequential(
            nn.Linear(4, 8),
            nn.ReLU()
        ).to(device_a)

        # Stage 2: lives on device_b (GPU)
        self.layer2 = nn.Linear(8, 1).to(device_b)

    def forward(self, x):
        # Input starts on device_a
        x = x.to(self.device_a)
        x = self.layer1(x)          # computed on CPU

        # --- Model-parallel handoff: move activations to the next device ---
        x = x.to(self.device_b)

        x = self.layer2(x)          # computed on GPU
        return x

model = TwoLayerModelParallelANN(device_a, device_b)
print(model)


TwoLayerModelParallelANN(
  (layer1): Sequential(
    (0): Linear(in_features=4, out_features=8, bias=True)
    (1): ReLU()
  )
  (layer2): Linear(in_features=8, out_features=1, bias=True)
)


## Step 3: Confirm the Split Actually Happened

Just to prove each layer really lives on the device we asked for (not just in theory).


In [4]:
print("Layer 1 weight device:", model.layer1[0].weight.device)
print("Layer 2 weight device:", model.layer2.weight.device)


Layer 1 weight device: cpu
Layer 2 weight device: cuda:0


## Step 4: Dummy Data

We'll make up a tiny toy regression problem: 4 input features -> 1 output number.
No real dataset needed, just random numbers, since the point here is to see the
**device split working**, not to solve a real task.


In [5]:
torch.manual_seed(0)

num_samples = 256
X = torch.randn(num_samples, 4)          # 256 samples, 4 features each
true_weights = torch.tensor([2.0, -1.0, 0.5, 3.0])
y = (X @ true_weights).unsqueeze(1) + 0.1 * torch.randn(num_samples, 1)  # + a little noise

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: torch.Size([256, 4])
y shape: torch.Size([256, 1])


## Step 5: Training Loop

Key thing to notice: the **loss must be computed on the same device as the final output**
(`device_b`, the GPU), so we move the target labels `y` there too before computing loss.
Everything else (optimizer step, backward pass) works automatically across both devices —
PyTorch's autograd tracks tensors across devices for you.


In [6]:
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

epochs = 200
for epoch in range(epochs):
    optimizer.zero_grad()

    predictions = model(X)                 # forward pass, crosses CPU -> GPU internally
    targets = y.to(device_b)               # move targets to the same device as the output

    loss = criterion(predictions, targets)
    loss.backward()                        # backward pass, crosses GPU -> CPU internally
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | Loss: {loss.item():.4f}")


Epoch  20/200 | Loss: 10.6018
Epoch  40/200 | Loss: 5.9849
Epoch  60/200 | Loss: 3.7723
Epoch  80/200 | Loss: 2.2455
Epoch 100/200 | Loss: 1.1338
Epoch 120/200 | Loss: 0.6060
Epoch 140/200 | Loss: 0.3811
Epoch 160/200 | Loss: 0.2642
Epoch 180/200 | Loss: 0.2012
Epoch 200/200 | Loss: 0.1650


## Step 6: Quick Sanity Check on a New Sample

Pass one new input through the model and see the model-parallel forward pass work end to end.


In [7]:
model.eval()
with torch.no_grad():
    sample = torch.tensor([[1.0, 0.5, -0.2, 2.0]])
    output = model(sample)
    print("Prediction:", output.item())
    print("Prediction was computed on device:", output.device)


Prediction: 6.204543113708496
Prediction was computed on device: cuda:0


## What Just Happened (Recap)

1. Layer 1's weights lived on the **CPU** the entire time.
2. Layer 2's weights lived on the **GPU** the entire time.
3. Every forward pass sent data: `CPU (Layer 1) -> GPU (Layer 2)`.
4. Every backward pass sent gradients back: `GPU (Layer 2) -> CPU (Layer 1)`.
5. No single device ever held the *whole* model's computation at once — this is Model Parallelism,
   just with only 2 tiny layers and 2 devices, so you can see the mechanics clearly.

**In a real large model**, this same `.to(device)` idea is used, just with many more layers and
many more GPUs, and usually wrapped by a library (like `torch.distributed`, DeepSpeed, or
Megatron-LM) instead of written by hand like we did here.

### If You Get a Second CUDA GPU Later

Just change:

```python
device_a = torch.device("cuda:0")
device_b = torch.device("cuda:1")
```

Nothing else in this notebook needs to change — the model and training loop are already written
to work with whatever two devices you assign to `device_a` and `device_b`.
